# OCO-2 and OCO-3 Lite SIF — Ogallala bounding box, native soundings, one CF file per mission
NASA catalogue → 10 parallel downloads → every variable decoded (corruption check) → soundings of the
aquifer's bounding box picked out by 6 dask workers → appended to a CF NetCDF → read back and compared →
source files deleted → repeat. Hourly verified copy to Drive, resumable from the Drive file.

**Data:** `OCO2_L2_Lite_SIF` (versions 11r and 11.2r) and `OCO3_L2_Lite_SIF` (version 11r), NASA JPL / Caltech,
distributed by NASA GES DISC. One file per day holds every sounding of the globe (up to about 270 thousand
soundings, 10 to 25 MB). As catalogued on 2026-10-07:

| Product | Version | Days with a file | Span |
|---|---|---|---|
| OCO-2 | 11r | 3,274 | 2014-09-06 → 2024-03-30 |
| OCO-2 | 11.2r | 818 | 2024-04-02 → 2026-07-27 |
| OCO-3 | 11r | 2,118 | 2019-08-06 → 2026-08-30 |

The files carry no spatial index, so every daily file is downloaded (about 120 GB in total) and filtered here.
OCO-2's record needs both versions; where both hold a day, the later version is used.

**Output** (`MyDrive/MSUGWB/OCO_SIF_LITE/`): `OCO2_L2_Lite_SIF_Ogallala.nc` and `OCO3_L2_Lite_SIF_Ogallala.nc`.
Each file has three record axes:

| Axis | One row per | Holds |
|---|---|---|
| `sounding` | sounding kept, in time order | every per-sounding variable NASA publishes (SIF at 740/757/771 nm, daily-corrected SIF, uncertainties, quality flags and bit flags, relative and unadjusted SIF, continuum radiances, footprint centre and 4 corners, sun and view angles, cloud screen values, meteorology, land fraction, IGBP class, orbit, footprint, mode, sounding id) |
| `day` | daily file processed, also when it has no sounding here | NASA's per-file tables (`Offset_*` statistics and histograms, `Diagnostics_*` thresholds, build labels) and the ledger: file name, version, build, size, counts |
| `sequence` | target / area-map sequence listed in a daily file | sequence name, id and mode |

Added per sounding: `day_index`, `source_index` (position in NASA's file), `collection_version`,
`in_polygon` (centre inside the aquifer polygon), `footprint_in_polygon` (footprint touches it), `sequence_row`.

**No-manipulation design**
- A sounding is kept when its centre, or any valid footprint corner, lies in the aquifer's bounding box.
  Nothing else is filtered: all quality flags, all measurement modes, cloudy soundings included.
- No gridding, averaging, gap filling, scaling or masking. Every value is the value in NASA's file, in its
  stored type, missing-value codes included. Quality flags are variables; they are not applied.
- A variable is stored under its path in NASA's file with the group folded into the name:
  `/Science/SIF_757nm` → `Science_SIF_757nm`. `source_variable` holds the original path.
- No `valid_range` attribute is written (readers would mask on it); NASA's is kept as `source_valid_range`.
- Variables are recognised by their axes, not by name, so one that NASA adds in a later build is carried
  along. Days whose file lacks a variable hold `_FillValue` for it.
- Cell 9 downloads a few stored days again and compares every stored value with NASA's file, bit for bit.

**Things to know about NASA's files** (measured on real files, not changed here)
- `Geolocation_time_tai93` is TAI. Decoded without a leap-second table it runs 8 to 10 s ahead of UTC.
- OCO-2 11r and 11.2r use different missing-value codes (−999 and −9e30) for four variables, and 11r lacks
  six variables that 11.2r has (and has one that 11.2r lacks). See `source_missing_values`.
- NASA has no file for some days: OCO-2 2017-07-31 → 2017-09-18 (50 days), OCO-3 2023-11-13 → 2024-07-15
  (246 days), and shorter gaps. The `day` axis shows which days exist; Cell 9 lists the gaps.
- `Offset_SIF_Mean_757nm` holds values near −3e12 in some files. They are NASA's and are kept.
- The root variables `Latitude`, `Longitude`, `SZA`, ... are NASA's own duplicates of `Geolocation_*`.

**Reading the files**
- `xarray.open_dataset(path)` works. It turns integer variables that have a `_FillValue` (flags, ids) into
  floats; use `mask_and_scale=False` to get the stored integers.
- Stop at `committed_soundings` / `committed_days` if `processing_status` is not `complete`.
- Typical filter: `Quality_Flag <= 1` (best and good), `in_polygon == 1`, and a choice of
  `Metadata_MeasurementMode` (0 nadir, 1 glint, 2 target, 3 area map, 4 transition).

**Before running**
1. An Earthdata Login account that has approved the application *NASA GESDISC DATA ARCHIVE*.
2. Colab Secrets `EARTHDATA_USERNAME` and `EARTHDATA_PASSWORD` (key icon in the left bar, notebook access on),
   or type them at the prompt in Cell 3.
3. `high_plains_quifer.zip` in `MyDrive/`.
4. `TEST_MODE = True` first: about 27 daily files into `.../_test/`, then a projection of the full run.
   Then set `TEST_MODE = False` and Run all. An interrupted run continues from the file on Drive.

In [ ]:
# ═══════════════ CELL 1: install (pinned) ═══════════════
# fsspec/s3fs/gcsfs are pinned to 2025.3.0: the newer fsspec that pip pulls in by default has produced
# truncated earthaccess downloads in this project (see OCO SIF.ipynb, Cell 2).
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q earthaccess netCDF4 geopandas shapely requests "dask[distributed]" "fsspec==2025.3.0" "s3fs==2025.3.0" "gcsfs==2025.3.0" 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE  = True              # True -> short rehearsal into .../_test/ ; False -> the whole record
TEST_DATES = {"OCO2": ("2024-03-25", "2024-04-08"),    # rehearsal window per mission (OCO-2: across the
              "OCO3": ("2019-08-06", "2019-08-20")}    # 11r -> 11.2r change of version)

MISSIONS   = ["OCO2", "OCO3"]  # one output file per mission
DATE_START = None              # None -> first day NASA holds; or "YYYY-MM-DD"
DATE_END   = None              # None -> latest day NASA holds; or "YYYY-MM-DD"
ROI_PAD_DEG = 0.0              # widen the aquifer's bounding box by this many degrees on every side

N_DOWNLOAD_THREADS = 10        # files downloaded at a time
DASK_WORKERS       = 6         # worker processes that verify the files and pick out the aquifer's soundings
WORKER_MEMORY_GB   = 1.5       # hard memory limit per dask worker
BATCH_DAYS         = 60        # daily files per cycle: download, extract, write, verify, delete (about 1.2 GB)
COMPRESS_LEVEL     = 1         # zlib level of the CF .nc (lossless)
SOUNDING_CHUNK     = 32768     # soundings per HDF5 chunk of the output

REPROCESS_CHANGED = False      # True -> rebuild from the first stored day whose file NASA has replaced or added
REBUILD           = False      # True -> ignore the files on Drive and start the outputs again

MIN_FREE_GB       = 20.0       # scratch space that must stay free at all times
DISK_MAX_WAIT_S   = 2 * 3600   # give up (cleanly, resumable) if space never comes back
BACKUP_INTERVAL_S = 3600       # copy the file in progress to Drive this often
MAX_RETRIES       = 4          # extra download attempts per file
CROSS_CHECKS      = 4          # stored days per mission that are downloaded again and compared value by value
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, product table, Earthdata login ═══════════════
import re, gc, sys, json, time, shutil, hashlib, logging, threading, random
from datetime import datetime, timezone, timedelta, date
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import netCDF4 as nc4
import geopandas as gpd
import shapely
import requests
import earthaccess
import dask
from dask.distributed import Client, LocalCluster

# force=True: Colab installs its own root log handler, so a plain basicConfig() is a
# no-op there and INFO progress lines would never be shown.
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("ocosif")
for _n in ("distributed", "urllib3.connectionpool", "pyogrio", "fiona", "earthaccess"):
    logging.getLogger(_n).setLevel(logging.WARNING)

# ---------- names and paths (Drive = durable, scratch = fast and disposable) ----------
ROI_LABEL  = "Ogallala"
DRIVE_ROOT = "/content/drive/MyDrive/MSUGWB"
DRIVE_OUT  = f"{DRIVE_ROOT}/OCO_SIF_LITE" + ("/_test" if TEST_MODE else "")
SCRATCH    = "/content/ocosif_scratch" + ("_test" if TEST_MODE else "")
RAW_DIR    = f"{SCRATCH}/raw"            # downloaded daily files (deleted after use)
NC_DIR     = f"{SCRATCH}/nc"             # the output files being built
DASK_TMP   = f"{SCRATCH}/dask"
DISK_PATH  = "/content"                  # same filesystem as scratch AND Drive's local upload cache
for _d in (DRIVE_OUT, RAW_DIR, NC_DIR, DASK_TMP):
    os.makedirs(_d, exist_ok=True)

# ---------- what NASA publishes ----------
# versions: oldest first. Where two versions hold a file for the same day, the later one in this list is used.
# OCO-2: 11r ends on 2024-03-30 and 11.2r starts on 2024-04-02, so the whole record needs both.
PRODUCTS = {
    "OCO2": dict(short_name="OCO2_L2_Lite_SIF", versions=["11r", "11.2r"], platform="OCO-2"),
    "OCO3": dict(short_name="OCO3_L2_Lite_SIF", versions=["11r"], platform="OCO-3 on the International Space Station"),
}
CMR_URL    = "https://cmr.earthdata.nasa.gov/search"
FILE_RE    = re.compile(r"(oco[23])_LtSIF_(\d{6})_(B\w+?)_(\d{12})s\.nc4$")
SOUND_DIM  = "sounding_dim"              # NASA's name of the per-sounding axis
SEQ_DIM    = "sequences_dim"             # NASA's name of the per-file list of target / area-map sequences
LAT, LON, TIME = "Geolocation/latitude", "Geolocation/longitude", "Geolocation/time_tai93"
CLAT, CLON = "Geolocation/footprint_latitude_vertices", "Geolocation/footprint_longitude_vertices"
SEQ_INDEX  = "Sequences/SequencesIndex"
# file attributes that change from day to day are kept per day; the others are copied once as source_*
GLOBALS_PER_DAY = ["product_version", "date_created", "gesdisc_collection", "InputBuildId", "InputCollectionLabel",
                   "date_time_coverage", "identifier_product_doi"]
GLOBALS_SKIP    = ["InputPointers", "day_of_year_coverage"]      # long per-day lists of upstream file names
DAY_EPOCH       = date(1970, 1, 1)
CF_VERSION      = "CF-1.9"               # first CF version that allows 64-bit integers and string variables


def out_name(mission):
    return f"{PRODUCTS[mission]['short_name']}_{ROI_LABEL}.nc"


def nc_name(path):
    """Output name of one of NASA's variables: its path with the group folded in, e.g.
    /Science/SIF_757nm -> Science_SIF_757nm. Mechanical and reversible, so nothing can collide
    (NASA's files hold both /Latitude and /Geolocation/latitude)."""
    return path.replace("/", "_")


# ---------- CF wording. Descriptive only: no stored value is touched ----------
UNITS_CF = {"W/m^2/sr/µm": "W m-2 sr-1 um-1", "W/m^2/sr/um": "W m-2 sr-1 um-1", "degrees": "degree",
            "degrees_north": "degrees_north", "degrees_east": "degrees_east", "Pa": "Pa", "K": "K", "m": "m",
            "m/s": "m s-1", "kg/kg": "kg kg-1", "per cent": "percent"}
UNITS_FORCED = {LON: "degrees_east",                                   # NASA's file gives this one no units
                TIME: "seconds since 1993-01-01 00:00:00",
                "Delta_Time": "seconds since 1990-01-01 00:00:00"}     # from NASA's description of the variable
STANDARD_NAMES = {LAT: "latitude", LON: "longitude", TIME: "time", "Geolocation/altitude": "surface_altitude"}
FLAGS = {   # from NASA's own description of each variable
    "Quality_Flag": ([-1, 0, 1, 2], "not_investigated best good bad"),
    "SimplyGoodOrBadQualityFlag": ([-1, 0, 1], "not_investigated good bad"),
    "Cloud/cloud_flag_abp": ([0, 1, 2], "classified_clear classified_cloudy not_classified"),
    "Metadata/MeasurementMode": ([0, 1, 2, 3, 4], "nadir glint target area_map transition"),
    "Science/sounding_l1b_quality_flag": ([0, 1], "good_l1b bad_l1b"),
    "Science/sounding_qual_flag": ([0, 1], "good bad"),
}
TIME_COMMENT = ("Copied unchanged. This is TAI: it counts every second since 1993-01-01, leap seconds included, "
                "so a clock time decoded from it without a leap-second table is ahead of UTC by 8 s (2014 to "
                "June 2015), 9 s (July 2015 to 2016) and 10 s (from 2017). Metadata_SoundingId holds the UTC "
                "time of the same sounding.")
COORDS_ATTR = f"{nc_name(TIME)} {nc_name(LAT)} {nc_name(LON)}"


class PipelineStop(RuntimeError):
    """Clean stop: everything written so far is safe, re-running resumes."""


class AuthError(PipelineStop):
    """Earthdata refused the login or the session."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)   # naive UTC everywhere


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")


def parse_day(s):
    return datetime.strptime(s, "%Y-%m-%d").date()


def day_number(d):
    return (d - DAY_EPOCH).days


def earthdata_login():
    """Log in without typing when Colab Secrets hold the credentials (needed for an
    unattended restart); otherwise fall back to ~/.netrc, then to the interactive prompt."""
    try:
        from google.colab import userdata
        for key in ("EARTHDATA_USERNAME", "EARTHDATA_PASSWORD", "EARTHDATA_TOKEN"):
            try:
                val = userdata.get(key)
                if val:
                    os.environ[key] = val
            except Exception:
                pass
    except Exception:
        pass
    if not (os.environ.get("EARTHDATA_TOKEN") or (os.environ.get("EARTHDATA_USERNAME")
                                                  and os.environ.get("EARTHDATA_PASSWORD"))):
        log.info("No Colab Secret / environment credentials - earthaccess will try ~/.netrc, then prompt.")
    auth = earthaccess.login()             # order: environment -> ~/.netrc -> interactive prompt
    if not getattr(auth, "authenticated", False):
        raise AuthError("Earthdata login failed - check EARTHDATA_USERNAME / EARTHDATA_PASSWORD.")
    log.info("Earthdata login OK")
    return auth


AUTH = earthdata_login()

In [ ]:
# ═══════════════ CELL 4: disk floor, verified Drive copy, logs ═══════════════
_LOG_LOCK   = threading.Lock()
_DRIVE_LOCK = threading.Lock()
DRIVE_SETTLE_S = 10                      # let the Drive FUSE mount settle before re-reading
TIMES = {}                               # seconds spent per stage (reported at the end of the run)
TOTALS = {}                              # counters (bytes downloaded, files, records) for the summary


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def log_events(name, entries):
    """Append JSON lines to a log on Drive in one write + fsync (survives a dead runtime)."""
    if not entries:
        return
    stamp = iso(utcnow())
    text = "".join(json.dumps(dict(e, logged=stamp), default=str) + "\n" for e in entries)
    with _LOG_LOCK:
        with open(os.path.join(DRIVE_OUT, name), "a") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


# ---------- 20 GB scratch floor ----------
def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. Scratch only ever holds the file being built,
    so a shortfall normally means Drive's local upload cache is still draining: wait for it
    rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for "
                               f"{DISK_MAX_WAIT_S // 60} min ({context}). Nothing is lost - re-run to resume.")
        time.sleep(60)


# ---------- verified copies to / from Drive ----------
def md5sum(path, block=10 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def robust_drive_copy(src, dst, max_retries=3):
    """Copy src to Drive under a temporary name, re-read it, compare size + MD5, then
    rename over dst. A runtime that dies mid-copy leaves the previous good dst intact."""
    if os.path.abspath(src) == os.path.abspath(dst):
        raise ValueError(f"robust_drive_copy: source and destination are the same file ({src})")
    with _DRIVE_LOCK, timed("Drive copy"):
        size = os.path.getsize(src)
        ensure_disk_free("Drive copy", extra_gb=size / 1024 ** 3)   # FUSE caches the upload locally
        want = md5sum(src)
        tmp = dst + ".tmp"
        os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)
        last = None
        for attempt in range(1, max_retries + 1):
            try:
                with open(src, "rb") as fs, open(tmp, "wb") as fd:
                    shutil.copyfileobj(fs, fd, length=10 * 1024 * 1024)
                    fd.flush()
                    os.fsync(fd.fileno())
                ok = False
                for _ in range(4):           # a just-closed FUSE file can return stale bytes
                    time.sleep(DRIVE_SETTLE_S)
                    try:
                        if os.path.getsize(tmp) == size and md5sum(tmp) == want:
                            ok = True
                            break
                    except OSError as e:
                        last = e
                if not ok:
                    raise IOError("size or MD5 mismatch after settle")
                os.replace(tmp, dst)
                log.info("Drive copy verified (%s): %s", human(size), os.path.basename(dst))
                return True
            except Exception as e:
                last = e
                log.warning("Drive copy attempt %d/%d failed: %s", attempt, max_retries, e)
                try:
                    if os.path.exists(tmp):
                        os.remove(tmp)
                except OSError:
                    pass
                if attempt < max_retries:
                    time.sleep(10 * attempt)
        raise RuntimeError(f"Drive copy of {os.path.basename(src)} failed after {max_retries} attempts: {last}")


def restore_from_drive(src, dst):
    """Bring a Drive file back to scratch (size-checked, atomic)."""
    size = os.path.getsize(src)
    ensure_disk_free("restore from Drive", extra_gb=size / 1024 ** 3)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    tmp = dst + ".part"
    shutil.copyfile(src, tmp)
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise IOError(f"restore of {os.path.basename(src)} is incomplete")
    os.replace(tmp, dst)
    log.info("Restored %s from Drive (%s)", os.path.basename(src), human(size))

In [ ]:
# ═══════════════ CELL 5: NASA's catalogue, verified downloads, the aquifer polygon ═══════════════
_tls = threading.local()


def _cmr_get(path, params, headers=None):
    """One catalogue request with retries. The catalogue is public: no login is sent."""
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            r = requests.get(f"{CMR_URL}/{path}", params=params, headers=headers or {}, timeout=(30, 180))
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            time.sleep(10 * attempt)
    raise PipelineStop(f"NASA's catalogue did not answer ({path}): {last}. Nothing is lost - re-run to resume.")


def list_versions(short_name):
    """{version: collection id} of every version of a product in NASA's catalogue."""
    feed = _cmr_get("collections.json", dict(short_name=short_name, page_size=50)).json()["feed"]["entry"]
    return {e["version_id"]: e["id"] for e in feed}


def list_granules(mission, version, concept_id):
    """Every daily file of one version: name, URL, day, build, production stamp."""
    out, after = [], None
    while True:
        r = _cmr_get("granules.umm_json", dict(collection_concept_id=concept_id, page_size=2000,
                                               sort_key="start_date"),
                     {"CMR-Search-After": after} if after else None)
        items = r.json().get("items", [])
        for it in items:
            umm = it["umm"]
            urls = [u["URL"] for u in umm.get("RelatedUrls", [])
                    if u.get("Type") == "GET DATA" and u["URL"].startswith("https://")]
            m = FILE_RE.search(urls[0]) if urls else None
            if not m:
                raise DataIntegrityError(f"Catalogue entry {umm.get('GranuleUR')} has no usable download link.")
            out.append(dict(name=os.path.basename(urls[0]), url=urls[0], mission=mission, version=version,
                            day=datetime.strptime(m.group(2), "%y%m%d").date(), build=m.group(3),
                            produced=m.group(4)))
        after = r.headers.get("CMR-Search-After")
        if not items or not after:
            break
    return out


def select_days(mission):
    """One file per day for a mission: the latest listed version that has the day, and within a version the
    most recently produced file. Returns (files in date order, number of days NASA holds in total)."""
    prod = PRODUCTS[mission]
    have = list_versions(prod["short_name"])
    extra = sorted(set(have) - set(prod["versions"]))
    if extra:
        log.warning("%s: NASA's catalogue also lists version(s) %s, which this notebook is not set to use "
                    "(see PRODUCTS in Cell 3).", mission, extra)
    best = {}
    for rank, version in enumerate(prod["versions"]):
        if version not in have:
            raise PipelineStop(f"{prod['short_name']} version {version} is not in NASA's catalogue any more. "
                               f"Versions on offer: {sorted(have)}. Update PRODUCTS in Cell 3.")
        grans = list_granules(mission, version, have[version])
        if not grans:
            log.warning("%s %s: the catalogue lists no file.", prod["short_name"], version)
            continue
        log.info("%s %s: %d daily files, %s to %s", prod["short_name"], version, len(grans),
                 min(g["day"] for g in grans), max(g["day"] for g in grans))
        for g in grans:
            g["rank"] = rank
            cur = best.get(g["day"])
            if cur is None or (g["rank"], g["produced"]) > (cur["rank"], cur["produced"]):
                best[g["day"]] = g
    days = sorted(best)
    if not days:
        raise PipelineStop(f"{mission}: NASA's catalogue lists no file at all.")
    lo, hi = (TEST_DATES[mission] if TEST_MODE else (DATE_START, DATE_END))
    lo = parse_day(lo) if lo else days[0]
    hi = parse_day(hi) if hi else days[-1]
    sel = [best[d] for d in days if lo <= d <= hi]
    if not sel:
        raise PipelineStop(f"{mission}: NASA holds no file between {lo} and {hi}.")
    return sel, len(days)


def calendar_gaps(days):
    """Runs of calendar days without a file: [(first, last, length), ...]."""
    return [(a + timedelta(days=1), b - timedelta(days=1), (b - a).days - 1)
            for a, b in zip(days, days[1:]) if (b - a).days > 1]


# ---------- downloads ----------
def _session():
    """One authenticated HTTPS session per download thread."""
    s = getattr(_tls, "session", None)
    if s is None:
        s = earthaccess.get_requests_https_session()
        _tls.session = s
    return s


def download_one(g, raw_dir=None):
    """Download one daily file and prove it is complete: byte count equals the server's Content-Length, the
    file starts like an HDF5 file, and its MD5 equals the server's when the server states one (an ETag with
    no part suffix). Every variable is decoded afterwards by a worker. Returns the server's file info."""
    dest = os.path.join(raw_dir or RAW_DIR, g["name"])
    part = dest + ".part"
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            with _session().get(g["url"], stream=True, timeout=(30, 300), allow_redirects=True) as r:
                if r.status_code in (401, 403):
                    raise AuthError(f"HTTP {r.status_code} for {g['name']}: Earthdata rejected the session. Check "
                                    f"the login, and that the account has approved 'NASA GESDISC DATA ARCHIVE'.")
                r.raise_for_status()
                ctype = (r.headers.get("Content-Type") or "").lower()
                if "html" in ctype:
                    raise IOError(f"the server sent {ctype} (a login page?), not data")
                expected = r.headers.get("Content-Length")
                etag = (r.headers.get("ETag") or "").strip('"')
                modified = r.headers.get("Last-Modified", "")
                with open(part, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        if chunk:
                            f.write(chunk)
                    f.flush()
                    os.fsync(f.fileno())
            got = os.path.getsize(part)
            if expected is not None and got != int(expected):
                raise IOError(f"incomplete: {got} of {expected} bytes")
            with open(part, "rb") as f:
                magic = f.read(4)
            if magic != b"\x89HDF":
                raise IOError(f"not an HDF5 / NetCDF-4 file (starts with {magic!r})")
            if re.fullmatch(r"[0-9a-f]{32}", etag) and md5sum(part) != etag:
                raise IOError("MD5 differs from the server's")
            os.replace(part, dest)
            return dict(name=g["name"], size=got, etag=etag, last_modified=modified)
        except AuthError:
            raise
        except Exception as e:
            last = e
            try:
                os.remove(part)
            except OSError:
                pass
            log.warning("download %s attempt %d/%d: %s", g["name"], attempt, MAX_RETRIES + 1, e)
            time.sleep(min(60, 5 * attempt))
    raise PipelineStop(f"{g['name']} could not be downloaded after {MAX_RETRIES + 1} attempts ({last}). "
                       f"Nothing is lost - re-run to resume.")


def download_group(grans):
    """Download a batch, N_DOWNLOAD_THREADS at a time, never eating into the scratch floor."""
    with timed("download"):
        ensure_disk_free("download", extra_gb=len(grans) * 0.03)       # a daily file is 10 to 25 MB
        t0 = time.time()
        with ThreadPoolExecutor(N_DOWNLOAD_THREADS) as ex:
            infos = list(ex.map(download_one, grans))
        dt = max(1e-6, time.time() - t0)
        size = sum(i["size"] for i in infos)
        log.info("Downloaded %d file(s), %s in %.0f s (%.1f MB/s)", len(grans), human(size), dt, size / 1e6 / dt)
        TOTALS["bytes_downloaded"] = TOTALS.get("bytes_downloaded", 0) + size
    return {i["name"]: i for i in infos}


def remove_raw(names, raw_dir=None):
    for n in names:
        for p in (os.path.join(raw_dir or RAW_DIR, n), os.path.join(raw_dir or RAW_DIR, n + ".part")):
            try:
                os.remove(p)
            except OSError:
                pass


# ---------- the aquifer polygon ----------
def load_roi():
    """The aquifer polygon (EPSG:4326, not simplified) and the bounding box whose soundings are kept."""
    gdf = gpd.read_file(SHP_PATH)
    if gdf.crs is None:
        raise DataIntegrityError("The aquifer shapefile has no CRS.")
    g4 = gdf.to_crs(epsg=4326).geometry
    poly = g4.union_all() if hasattr(g4, "union_all") else g4.unary_union
    if not poly.is_valid:
        poly = poly.buffer(0)
    w, s, e, n = poly.bounds
    bbox = (w - ROI_PAD_DEG, s - ROI_PAD_DEG, e + ROI_PAD_DEG, n + ROI_PAD_DEG)
    log.info("Aquifer polygon: %d vertices | bounding box kept: lon %.4f to %.4f, lat %.4f to %.4f",
             shapely.get_num_coordinates(poly), bbox[0], bbox[2], bbox[1], bbox[3])
    return dict(poly=poly, bbox=tuple(float(v) for v in bbox), wkb=shapely.to_wkb(poly))

In [ ]:
# ═══════════════ CELL 6: verify one daily file and pick out the aquifer's soundings (runs in dask workers) ═══════════════
def extract_file(path, spec, roi_wkb):
    """Decode EVERY variable of one daily file (a damaged file fails here), then keep the soundings whose
    centre or any footprint corner lies in the bounding box. Values are returned exactly as stored: no
    scaling, masking, casting or filtering. Returns plain numpy / Python objects.

    The file's variables fall into three kinds, recognised by their axes, not by name, so a variable NASA
    adds in a later build is carried along without a change here:
      sounding : first axis is the per-sounding axis             -> subset to the kept soundings
      sequence : along the file's list of target / area-map runs -> kept whole
      day      : everything else (per-file tables and labels)    -> kept whole
    """
    import os
    import numpy as np
    import netCDF4 as nc4
    import shapely

    out = dict(name=os.path.basename(path), ok=False)
    try:
        def plain(v):
            if isinstance(v, np.ndarray):
                return v.tolist() if v.size != 1 else v.reshape(-1)[0].item()
            if isinstance(v, np.generic):
                return v.item()
            return v

        def walk(g, prefix=""):
            for n, v in g.variables.items():
                yield prefix + n, v
            for n, sub in g.groups.items():
                yield from walk(sub, prefix + n + "/")

        meta, sounding, daily, seq = {}, {}, {}, {}
        with nc4.Dataset(path) as ds:
            ds.set_auto_maskandscale(False)                      # raw stored values, in every group
            if spec["sound_dim"] not in ds.dimensions:
                raise ValueError("the file has no per-sounding axis")
            n_file = ds.dimensions[spec["sound_dim"]].size
            dim_sizes = {k: int(d.size) for k, d in ds.dimensions.items()}
            glob = {k: str(ds.getncattr(k)) for k in ds.ncattrs() if k not in spec["globals_skip"]}
            arrays = {}
            for p, v in walk(ds):
                attrs = {a: plain(v.getncattr(a)) for a in v.ncattrs()}
                if float(attrs.get("scale_factor", 1)) != 1.0 or float(attrs.get("add_offset", 0)) != 0.0:
                    raise ValueError(f"{p} is packed (scale_factor/add_offset), which this notebook does not expect")
                is_str = v.dtype is str
                dims = tuple(v.dimensions)
                if dims and dims[0] == spec["sound_dim"]:
                    kind = "sounding"
                elif dims == (spec["seq_dim"],):
                    kind = "sequence"
                elif spec["sound_dim"] in dims or spec["seq_dim"] in dims:
                    raise ValueError(f"{p} has an unexpected shape {dims}")
                else:
                    kind = "day"
                if is_str and kind == "sounding":
                    raise ValueError(f"{p} is a per-sounding text variable, which this notebook does not expect")
                a = v[...]                                       # full read = decode check of every chunk
                if is_str:
                    a = [str(x) for x in np.asarray(a, dtype=object).reshape(-1)]
                    if not dims:
                        a = a[0]
                else:
                    a = np.asarray(a)
                    if a.dtype != v.dtype:
                        raise ValueError(f"{p} was not read in its stored type")
                meta[p] = dict(kind=kind, dtype="str" if is_str else v.dtype.str, attrs=attrs,
                               dims=[(d, dim_sizes[d]) for d in dims if d not in (spec["sound_dim"], spec["seq_dim"])])
                arrays[p] = a

        for need in (spec["lat"], spec["lon"], spec["time"], spec["clat"], spec["clon"]):
            if need not in arrays:
                raise ValueError(f"{need} is missing from the file")
        lat, lon, t = arrays[spec["lat"]], arrays[spec["lon"]], arrays[spec["time"]]
        clat, clon = arrays[spec["clat"]], arrays[spec["clon"]]
        w, s, e, n = spec["bbox"]
        centre = (lat >= s) & (lat <= n) & (lon >= w) & (lon <= e)
        corner = ((clat >= s) & (clat <= n) & (clon >= w) & (clon <= e)).any(axis=1)
        sel = np.flatnonzero(centre | corner)
        sel = sel[np.argsort(t[sel], kind="stable")]             # time order; ties keep the file's order
        m = int(sel.size)

        in_poly = np.zeros(m, "i1")
        fp_poly = np.full(m, -1, "i1")                           # -1: a corner is missing, so not decidable
        if m:
            la, lo, tt = lat[sel].astype("f8"), lon[sel].astype("f8"), t[sel]
            if not (np.all(np.isfinite(la)) and np.all(np.isfinite(lo)) and np.all(np.isfinite(tt))
                    and np.all(np.abs(la) <= 90) and np.all(np.abs(lo) <= 180) and np.all(tt > 0)):
                raise ValueError("a kept sounding has no usable position or time")
            roi = shapely.from_wkb(roi_wkb)
            shapely.prepare(roi)
            in_poly = shapely.contains_xy(roi, lo, la).astype("i1")
            cla, clo = clat[sel].astype("f8"), clon[sel].astype("f8")
            good = np.all(np.isfinite(cla) & np.isfinite(clo) & (np.abs(cla) <= 90) & (np.abs(clo) <= 180), axis=1)
            if good.any():
                # the hull of the four corners is the footprint whatever order NASA lists them in
                pts = shapely.multipoints(np.stack([clo[good], cla[good]], axis=-1))
                fp_poly[good] = shapely.intersects(roi, shapely.convex_hull(pts)).astype("i1")

        for p, a in arrays.items():
            k = meta[p]["kind"]
            if k == "sounding":
                if a.shape[0] != n_file:
                    raise ValueError(f"{p} has {a.shape[0]} rows, the file has {n_file} soundings")
                sounding[p] = np.ascontiguousarray(a[sel])
            elif k == "sequence":
                seq[p] = a
            else:
                daily[p] = a
        n_seq = {len(v) for v in seq.values()}
        if len(n_seq) > 1:
            raise ValueError("the sequence lists of the file have different lengths")

        out.update(ok=True, n_file=int(n_file), n_kept=m, n_centre_in_bbox=int(centre.sum()),
                   n_in_polygon=int(in_poly.sum()), n_sequences=int(n_seq.pop()) if n_seq else 0,
                   meta=meta, sounding=sounding, day=daily, sequence=seq, globals=glob,
                   source_index=sel.astype("i4"), in_polygon=in_poly, footprint_in_polygon=fp_poly)
    except Exception as e:
        out["error"] = f"{type(e).__name__}: {e}"
    return out


def worker_spec(roi):
    return dict(sound_dim=SOUND_DIM, seq_dim=SEQ_DIM, lat=LAT, lon=LON, time=TIME, clat=CLAT, clon=CLON,
                bbox=tuple(roi["bbox"]), globals_skip=list(GLOBALS_SKIP))


def start_cluster():
    dask.config.set({"temporary-directory": DASK_TMP, "distributed.worker.daemon": False})
    cluster = LocalCluster(n_workers=DASK_WORKERS, threads_per_worker=1, processes=True,
                           memory_limit=f"{WORKER_MEMORY_GB}GB", dashboard_address=None)
    client = Client(cluster)
    log.info("dask: %d workers x %.1f GB", DASK_WORKERS, WORKER_MEMORY_GB)
    return cluster, client


def extract_group(client, grans, roi_future, spec, raw_dir=None):
    """Run extract_file on a batch. A file that does not decode is downloaded once more; if it fails again
    the run stops (a day is never skipped silently). Returns the results in the order of `grans`."""
    with timed("verify + extract"):
        def run(batch):
            futs = [client.submit(extract_file, os.path.join(raw_dir or RAW_DIR, g["name"]), spec, roi_future,
                                  pure=False) for g in batch]
            return client.gather(futs)

        results = run(grans)
        bad = [i for i, r in enumerate(results) if not r["ok"]]
        if bad:
            for i in bad:
                log.warning("%s did not pass the decode check (%s) - downloading it again",
                            grans[i]["name"], results[i]["error"])
                remove_raw([grans[i]["name"]], raw_dir)
                download_one(grans[i], raw_dir)
            for i, r in zip(bad, run([grans[i] for i in bad])):
                if not r["ok"]:
                    raise PipelineStop(f"{grans[i]['name']} is unreadable after two downloads ({r['error']}). "
                                       f"Nothing is lost - re-run to resume; if it persists the file is damaged "
                                       f"at NASA.")
                results[i] = r
    return results

In [ ]:
# ═══════════════ CELL 7: the CF output file (create, add variables, append a batch, read back) ═══════════════
# Layout of one output file (one per mission). Three record axes, each with its own unlimited dimension:
#   sounding : one row per sounding kept (the bounding box of the aquifer), in time order
#   day      : one row per daily file processed, including days with no sounding over the aquifer.
#              Holds NASA's per-file tables (Offset/*, Diagnostics thresholds, build labels) and the ledger.
#   sequence : one row per target / area-map sequence listed in a daily file
# Every one of NASA's variables keeps its stored type and its stored values, missing-value codes included.
_COMP = dict(zlib=True, complevel=COMPRESS_LEVEL, shuffle=True)


def raw(ds):
    """Never let netCDF4 mask or scale: this notebook only moves stored values."""
    ds.set_auto_maskandscale(False)
    return ds


def _var(ds, name, dtype, dims, fill=None, chunk=None, **attrs):
    if dtype is str:
        v = ds.createVariable(name, str, dims, chunksizes=chunk)
    else:
        v = ds.createVariable(name, dtype, dims, fill_value=fill, chunksizes=chunk, **_COMP)
    v.setncatts({k: a for k, a in attrs.items() if a is not None})
    return v


def create_output(path, mission, roi):
    prod = PRODUCTS[mission]
    w, s, e, n = roi["bbox"]
    ds = raw(nc4.Dataset(path, "w", format="NETCDF4"))
    for d in ("sounding", "day", "sequence"):
        ds.createDimension(d, None)
    ds.setncatts(dict(
        Conventions=CF_VERSION, featureType="point", product_kind="oco_sif_lite", mission=mission,
        title=f"{prod['platform']} Level 2 Lite solar-induced fluorescence soundings over the bounding box of the "
              f"{ROI_LABEL} (High Plains) aquifer",
        summary="Every sounding of NASA's daily Lite SIF files whose centre or any footprint corner lies in the "
                "bounding box of the aquifer, with every variable NASA publishes, at native footprint resolution. "
                "No gridding, averaging, gap filling, scaling, quality filtering or masking: each value is the "
                "value stored in NASA's file. in_polygon and footprint_in_polygon say where a sounding lies "
                "relative to the aquifer polygon.",
        institution="Data: NASA Jet Propulsion Laboratory / Caltech, distributed by NASA GES DISC. "
                    "Subset: MSU groundwater project.",
        source=f"{prod['short_name']} version(s) {', '.join(prod['versions'])}, daily files downloaded from "
               f"https://data.gesdisc.earthdata.nasa.gov",
        platform=prod["platform"], product_short_name=prod["short_name"],
        product_versions=" ".join(prod["versions"]),
        version_rule="Where two versions hold a file for the same day the later version is used; "
                     "collection_version and day_collection_version say which one each row comes from.",
        selection_rule="centre inside the bounding box, or at least one valid footprint corner inside it",
        roi_name=f"{ROI_LABEL} / High Plains aquifer (hp_bound2010)", roi_pad_degrees=float(ROI_PAD_DEG),
        geospatial_lat_min=float(s), geospatial_lat_max=float(n),
        geospatial_lon_min=float(w), geospatial_lon_max=float(e),
        variable_naming="A variable of NASA's file is stored under its path with the group folded into the "
                        "name: /Science/SIF_757nm is Science_SIF_757nm. source_variable holds the original path. "
                        "Variables starting with day_ or sequence_, and day_index, source_index, "
                        "collection_version, in_polygon, footprint_in_polygon and sequence_row, are added here.",
        missing_data="Each variable keeps NASA's missing-value code as _FillValue. Where NASA's versions use "
                     "different codes for one variable, all of them are listed in source_missing_values and "
                     "each is left in the data as stored. Rows of a day whose file lacks a variable hold "
                     "_FillValue.",
        history=f"{iso(utcnow())} created by OCO_SIF_Lite_Ogallala.ipynb",
        date_created=iso(utcnow()), processing_status="in_progress",
        committed_soundings=np.int64(0), committed_days=np.int32(0), committed_sequences=np.int32(0)))
    c1 = (SOUNDING_CHUNK,)
    flags = np.arange(len(prod["versions"]), dtype="i1")
    _var(ds, "day_index", "i4", ("sounding",), chunk=c1, long_name="Row of the day axis this sounding comes from")
    _var(ds, "source_index", "i4", ("sounding",), chunk=c1,
         long_name="Position of this sounding on the sounding axis of NASA's daily file (zero-based)")
    _var(ds, "collection_version", "i1", ("sounding",), fill=np.int8(-127), chunk=c1,
         long_name="Product version of NASA's file this sounding comes from", flag_values=flags,
         flag_meanings=" ".join("v" + v for v in prod["versions"]), coordinates=COORDS_ATTR)
    _var(ds, "in_polygon", "i1", ("sounding",), fill=np.int8(-127), chunk=c1,
         long_name="Centre of the sounding lies inside the aquifer polygon", flag_values=np.array([0, 1], "i1"),
         flag_meanings="outside inside", coordinates=COORDS_ATTR)
    _var(ds, "footprint_in_polygon", "i1", ("sounding",), fill=np.int8(-127), chunk=c1,
         long_name="Footprint (hull of the four corners) intersects the aquifer polygon",
         flag_values=np.array([-1, 0, 1], "i1"), flag_meanings="corner_missing no yes", coordinates=COORDS_ATTR)
    _var(ds, "sequence_row", "i4", ("sounding",), chunk=c1,
         long_name="Row of the sequence axis for a sounding that belongs to a target or area-map sequence",
         comment="-1: the sounding belongs to no sequence")
    cd = (512,)
    _var(ds, "day_time", "i4", ("day",), chunk=cd, long_name="Date of NASA's daily file", standard_name="time",
         units="days since 1970-01-01 00:00:00", calendar="standard")
    for name, text in (("day_source_file", "Name of NASA's daily file"),
                       ("day_collection_version", "Product version of the daily file"),
                       ("day_build", "Build label in the name of the daily file"),
                       ("day_production_stamp", "Production stamp in the name of the daily file (YYMMDDhhmmss)"),
                       ("day_source_etag", "ETag the server sent with the daily file"),
                       ("day_source_last_modified", "Last-Modified date the server sent with the daily file")):
        _var(ds, name, str, ("day",), chunk=cd, long_name=text)
    _var(ds, "day_source_size", "i8", ("day",), chunk=cd, long_name="Size of the daily file", units="byte")
    for name, text in (("day_n_soundings_file", "Soundings in NASA's daily file (whole globe)"),
                       ("day_n_soundings", "Soundings of the day kept here"),
                       ("day_n_in_polygon", "Soundings of the day whose centre is inside the aquifer polygon"),
                       ("day_n_sequences", "Sequences listed in the daily file")):
        _var(ds, name, "i4", ("day",), chunk=cd, long_name=text, units="1")
    _var(ds, "day_first_sounding", "i8", ("day",), chunk=cd,
         long_name="Row of the sounding axis where the soundings of this day start")
    _var(ds, "day_first_sequence", "i4", ("day",), chunk=cd,
         long_name="Row of the sequence axis where the sequences of this day start")
    for a in GLOBALS_PER_DAY:
        _var(ds, "day_global_" + a, str, ("day",), chunk=cd, long_name=f"Attribute {a} of NASA's daily file")
    _var(ds, "sequence_day_index", "i4", ("sequence",), chunk=cd, long_name="Row of the day axis of this sequence")
    _var(ds, "sequence_local_index", "i4", ("sequence",), chunk=cd,
         long_name="Position of this sequence in the list of NASA's daily file (zero-based), the value "
                   "Sequences_SequencesIndex refers to")
    ds.sync()
    return ds


def _axis_of(v):
    return v.dimensions[0] if v.dimensions else None


def _default_fill(dt):
    return np.array(nc4.default_fillvals[dt.str[1:]], dtype=dt)[()]


def ensure_var(ds, path, m, source_file):
    """Create the output variable for one of NASA's variables on first sight, in its stored type, with its
    missing-value code as _FillValue. On later sight: the type and shape must be the same."""
    name = nc_name(path)
    axis = {"sounding": "sounding", "day": "day", "sequence": "sequence"}[m["kind"]]
    dims = (axis,) + tuple(d for d, _ in m["dims"])
    is_str = m["dtype"] == "str"
    dt = None if is_str else np.dtype(m["dtype"])
    a = m["attrs"]
    code = None
    if not is_str and a.get("missing_value") is not None:
        code = np.array(a["missing_value"], dtype=dt)[()]
    if name in ds.variables:
        v = ds[name]
        have = "str" if v.dtype is str else np.dtype(v.dtype).str
        if have != m["dtype"] or tuple(v.dimensions) != dims:
            raise DataIntegrityError(f"{path} in {source_file} is {m['dtype']} {dims}; the output holds it as "
                                     f"{have} {tuple(v.dimensions)}. NASA changed the variable - not merged.")
        if code is not None and code != v.getncattr("_FillValue"):
            # another version of the product uses another code for this variable: both stay in the data as
            # stored, and both are named here (CF allows only one code in _FillValue / missing_value)
            known = {v.getncattr("_FillValue").item()}
            if "source_missing_values" in v.ncattrs():
                known = set(np.atleast_1d(v.getncattr("source_missing_values")).tolist())
            if code.item() not in known:
                v.source_missing_values = np.array(sorted(known | {code.item()}), dtype=dt)
        return v
    for d, size in m["dims"]:
        if d not in ds.dimensions:
            ds.createDimension(d, size)
        elif ds.dimensions[d].size != size:
            raise DataIntegrityError(f"{path} in {source_file}: axis {d} has {size} entries, the output has "
                                     f"{ds.dimensions[d].size}.")
    tail = tuple(size for _, size in m["dims"])
    cells = int(np.prod(tail)) if tail else 1
    if axis == "sounding":
        chunk = (max(1, SOUNDING_CHUNK // cells),) + tail
    else:
        chunk = (max(1, min(512, (1 << 18) // max(1, cells * (8 if is_str else dt.itemsize)))),) + tail
    src_units = a.get("units")
    attrs = dict(long_name=a.get("long_name"), description=a.get("description"), source_variable="/" + path)
    if is_str:
        v = _var(ds, name, str, dims, chunk=chunk, **attrs)
    else:
        fill = code if code is not None else _default_fill(dt)
        units = UNITS_FORCED.get(path) or UNITS_CF.get(src_units)
        if units is None and dt.kind == "f" and src_units in (None, "n/a"):
            units = "1"
        if units is None and src_units not in (None, "n/a", "YYYYMMDDHHMMSS") and path not in FLAGS:
            units = src_units                                   # an unforeseen unit string: keep NASA's wording
        attrs.update(units=units, standard_name=STANDARD_NAMES.get(path) or
                     {"degrees_north": "latitude", "degrees_east": "longitude"}.get(units))
        if path in (TIME, "Delta_Time"):
            attrs.update(calendar="standard", comment=TIME_COMMENT)
        if path in FLAGS:
            attrs.update(flag_values=np.array(FLAGS[path][0], dtype=dt), flag_meanings=FLAGS[path][1])
        if axis == "sounding" and path not in (LAT, LON, TIME):
            attrs["coordinates"] = COORDS_ATTR
        v = _var(ds, name, dt, dims, fill=fill, chunk=chunk, **attrs)
    extra = {}
    if src_units is not None and src_units != v.__dict__.get("units"):
        extra["source_units"] = src_units
    for k in ("valid_range", "least_significant_digit", "standard_name", "coordinates"):
        if k in a and not (k == "standard_name" and a[k] == v.__dict__.get("standard_name")):
            # kept for reference under another name: netCDF4 and xarray act on valid_range by masking
            extra["source_" + k] = np.array(a[k], dtype=dt) if (k == "valid_range" and not is_str) else a[k]
    v.setncatts(extra)
    return v


def _fill_block(v, shape):
    if v.dtype is str:
        return np.full(shape, "", dtype=object)
    return np.full(shape, v.getncattr("_FillValue"), dtype=v.dtype)


def _same(got, want):
    """Bitwise comparison (so NaN equals NaN and -0.0 differs from 0.0)."""
    if isinstance(want, np.ndarray) and want.dtype != object:
        got = np.asarray(got)
        return got.dtype == want.dtype and got.shape == want.shape and got.tobytes() == want.tobytes()
    return [str(x) for x in np.asarray(got, dtype=object).reshape(-1)] == \
           [str(x) for x in np.asarray(want, dtype=object).reshape(-1)]


def write_batch(ds, mission, grans, infos, results, state):
    """Append a batch of days (in date order) to the three axes, then read every written block back and
    compare it bitwise with what was meant to be written. Returns the new (soundings, days, sequences)."""
    n0, d0, q0 = state["n"], state["d"], state["q"]
    K = len(results)
    with timed("write"):
        for g, r in zip(grans, results):
            for path, m in r["meta"].items():
                ensure_var(ds, path, m, g["name"])
        counts = np.array([r["n_kept"] for r in results], "i8")
        nseq = np.array([r["n_sequences"] for r in results], "i8")
        first = n0 + np.concatenate([[0], np.cumsum(counts)[:-1]])
        firstq = q0 + np.concatenate([[0], np.cumsum(nseq)[:-1]])
        M, Q = int(counts.sum()), int(nseq.sum())
        written = []

        def put(name, lo, block):
            ds[name][lo:lo + len(block)] = block
            written.append((name, lo, block))

        # ---- sounding axis ----
        if M:
            own = dict(
                day_index=np.repeat(np.arange(d0, d0 + K, dtype="i4"), counts),
                source_index=np.concatenate([r["source_index"] for r in results]).astype("i4"),
                collection_version=np.repeat(np.array([g["rank"] for g in grans], "i1"), counts),
                in_polygon=np.concatenate([r["in_polygon"] for r in results]).astype("i1"),
                footprint_in_polygon=np.concatenate([r["footprint_in_polygon"] for r in results]).astype("i1"))
            rows = []
            for r, fq in zip(results, firstq):
                idx = r["sounding"].get(SEQ_INDEX)
                row = np.full(r["n_kept"], -1, "i4")
                if idx is not None and r["n_kept"]:
                    inseq = idx >= 0
                    if np.any(idx[inseq] >= r["n_sequences"]):
                        raise DataIntegrityError(f"{r['name']}: a sounding points to a sequence the file does not list.")
                    row[inseq] = (fq + idx[inseq]).astype("i4")
                rows.append(row)
            own["sequence_row"] = np.concatenate(rows)
            for name, block in own.items():
                put(name, n0, block)
            for name, v in ds.variables.items():
                if _axis_of(v) != "sounding" or "source_variable" not in v.ncattrs():
                    continue
                path = v.source_variable[1:]
                tail = v.shape[1:]
                block = np.concatenate([r["sounding"][path] if path in r["sounding"]
                                        else _fill_block(v, (r["n_kept"],) + tail) for r in results])
                put(name, n0, block)
            t = np.concatenate([r["sounding"][TIME] for r in results])
            if np.any(np.diff(t) < 0) or (state.get("last_time") is not None and t[0] < state["last_time"]):
                log.warning("%s: soundings of %s to %s are not in time order across files.", mission,
                            grans[0]["day"], grans[-1]["day"])
            state["last_time"] = float(t.max())
        # ---- sequence axis ----
        if Q:
            put("sequence_day_index", q0, np.repeat(np.arange(d0, d0 + K, dtype="i4"), nseq))
            put("sequence_local_index", q0, np.concatenate([np.arange(k, dtype="i4") for k in nseq]))
            for name, v in ds.variables.items():
                if _axis_of(v) != "sequence" or "source_variable" not in v.ncattrs():
                    continue
                path = v.source_variable[1:]
                parts = []
                for r in results:
                    if path in r["sequence"]:
                        x = r["sequence"][path]
                        parts.append(np.array(x, dtype=object) if v.dtype is str else np.asarray(x))
                    else:
                        parts.append(_fill_block(v, (r["n_sequences"],)))
                put(name, q0, np.concatenate(parts) if parts else _fill_block(v, (0,)))
        # ---- day axis ----
        obj = lambda xs: np.array(list(xs), dtype=object)
        put("day_time", d0, np.array([day_number(g["day"]) for g in grans], "i4"))
        put("day_source_file", d0, obj(g["name"] for g in grans))
        put("day_collection_version", d0, obj(g["version"] for g in grans))
        put("day_build", d0, obj(g["build"] for g in grans))
        put("day_production_stamp", d0, obj(g["produced"] for g in grans))
        put("day_source_etag", d0, obj(infos[g["name"]]["etag"] for g in grans))
        put("day_source_last_modified", d0, obj(infos[g["name"]]["last_modified"] for g in grans))
        put("day_source_size", d0, np.array([infos[g["name"]]["size"] for g in grans], "i8"))
        put("day_n_soundings_file", d0, np.array([r["n_file"] for r in results], "i4"))
        put("day_n_soundings", d0, counts.astype("i4"))
        put("day_n_in_polygon", d0, np.array([r["n_in_polygon"] for r in results], "i4"))
        put("day_n_sequences", d0, nseq.astype("i4"))
        put("day_first_sounding", d0, first.astype("i8"))
        put("day_first_sequence", d0, firstq.astype("i4"))
        for a in GLOBALS_PER_DAY:
            put("day_global_" + a, d0, obj(r["globals"].get(a, "") for r in results))
        for name, v in ds.variables.items():
            if _axis_of(v) != "day" or "source_variable" not in v.ncattrs():
                continue
            path = v.source_variable[1:]
            tail = v.shape[1:]
            if v.dtype is str:
                put(name, d0, obj(r["day"].get(path, "") for r in results))
            else:
                put(name, d0, np.stack([r["day"][path] if path in r["day"] else _fill_block(v, tail)
                                        for r in results]))
        if "source_attributes_from" not in ds.ncattrs():          # the file-level attributes, copied once
            ds.setncatts({"source_" + k: val for k, val in results[0]["globals"].items()
                          if k not in GLOBALS_PER_DAY})
            ds.source_attributes_from = grans[0]["name"]
        ds.sync()
    with timed("read-back check"):
        for name, lo, block in written:
            if not _same(ds[name][lo:lo + len(block)], block):
                raise DataIntegrityError(f"{name}: rows {lo} to {lo + len(block) - 1} read back differently from "
                                         f"what was written. The output on scratch is not trusted; the copy on "
                                         f"Drive is untouched.")
    return n0 + M, d0 + K, q0 + Q


def commit(ds, n, d, q, complete):
    """Move the commit markers. Rows beyond them are never read by this notebook."""
    ds.committed_soundings, ds.committed_days, ds.committed_sequences = np.int64(n), np.int32(d), np.int32(q)
    ds.date_modified = iso(utcnow())
    if d:
        t = ds["day_time"][:d]
        ds.time_coverage_start = (DAY_EPOCH + timedelta(days=int(t[0]))).isoformat()
        ds.time_coverage_end = (DAY_EPOCH + timedelta(days=int(t[-1]))).isoformat()
    ds.processing_status = "complete" if complete else "in_progress"
    ds.sync()


def read_state(path):
    """What the output already holds: commit markers and the file name stored for each day."""
    with nc4.Dataset(path) as ds:
        raw(ds)
        if str(getattr(ds, "product_kind", "")) != "oco_sif_lite":
            raise PipelineStop(f"{path} is not an output of this notebook.")
        d = min(int(ds.committed_days), ds.dimensions["day"].size)
        return dict(mission=str(ds.mission), d=d, n=int(ds.committed_soundings), q=int(ds.committed_sequences),
                    bbox=(float(ds.geospatial_lon_min), float(ds.geospatial_lat_min),
                          float(ds.geospatial_lon_max), float(ds.geospatial_lat_max)),
                    files=[str(x) for x in ds["day_source_file"][:d]],
                    days=[DAY_EPOCH + timedelta(days=int(x)) for x in ds["day_time"][:d]],
                    first_sounding=np.asarray(ds["day_first_sounding"][:d], "i8"),
                    first_sequence=np.asarray(ds["day_first_sequence"][:d], "i8"),
                    sizes=dict(sounding=ds.dimensions["sounding"].size, day=ds.dimensions["day"].size,
                               sequence=ds.dimensions["sequence"].size))


def _committed_days(path):
    try:
        with nc4.Dataset(path) as ds:
            return int(ds.committed_days)
    except Exception:
        return -1


def backup(local, drive_path):
    """Verified copy of the output to Drive (temporary name, MD5, then rename)."""
    robust_drive_copy(local, drive_path)


def stage_output(name):
    """Put the output to continue from on scratch: the copy on Drive, unless scratch already holds a
    readable copy that is further along (a run interrupted before its next backup)."""
    local, on_drive = os.path.join(NC_DIR, name), os.path.join(DRIVE_OUT, name)
    if REBUILD:
        if os.path.exists(local):
            os.remove(local)
        log.info("REBUILD: %s starts again (the copy on Drive is replaced at the first backup).", name)
        return local
    have_local = _committed_days(local) if os.path.exists(local) else -1
    have_drive = _committed_days(on_drive) if os.path.exists(on_drive) else -1
    if os.path.exists(on_drive) and have_drive < 0:
        raise PipelineStop(f"{on_drive} exists but cannot be read. Move it away, or set REBUILD = True.")
    if have_drive >= 0 and have_drive >= have_local:
        restore_from_drive(on_drive, local)
    elif have_local >= 0:
        log.info("Continuing from the copy on scratch (%d days; Drive has %d).", have_local, max(have_drive, 0))
    elif os.path.exists(local):
        os.remove(local)                                         # unreadable leftover
    return local

In [ ]:
# ═══════════════ CELL 8: one mission, batch by batch, then RUN ═══════════════
def resume_point(st, sel):
    """Where to continue. The stored days are kept while they are, in order, the files NASA's catalogue
    selects today. Returns (days to keep, index in `sel` to continue from)."""
    names = [g["name"] for g in sel]
    d = st["d"]
    k = 0
    while k < d and k < len(names) and st["files"][k] == names[k]:
        k += 1
    if k == d:
        return d, d                                   # everything stored is current: append what is new
    if k == len(names):
        log.info("The file holds %d days beyond the date window of this run; nothing is removed.", d - k)
        return d, len(names)
    stored, wanted = set(st["files"]), set(names)
    last = st["days"][-1]
    replaced = sum(1 for f in st["files"] if f not in wanted)
    added = sum(1 for g in sel if g["day"] <= last and g["name"] not in stored)
    if st["days"][0] != sel[0]["day"] and not REPROCESS_CHANGED:
        raise PipelineStop(f"The file on Drive starts on {st['days'][0]} but this run starts on {sel[0]['day']}. "
                           f"Restore DATE_START, or set REBUILD = True.")
    if REPROCESS_CHANGED:
        log.warning("NASA's catalogue differs from the stored days from %s on (%d stored file(s) are no longer "
                    "the ones it selects, %d selected file(s) up to the last stored day are not stored). "
                    "REPROCESS_CHANGED: rebuilding from that day.", sel[k]["day"], replaced, added)
        return k, k
    log.warning("NASA's catalogue differs from the stored days from %s on: %d stored file(s) are no longer the ones "
                "it selects and %d selected file(s) up to the last stored day are not stored. The stored days are "
                "left as they are. Set REPROCESS_CHANGED = True to rebuild from that day.", sel[k]["day"], replaced,
                added)
    return d, next((i for i, g in enumerate(sel) if g["day"] > last), len(sel))


def run_mission(client, roi, roi_future, mission):
    """Download a batch of daily files, verify, keep the aquifer's soundings, append, read back, delete, repeat."""
    sel, n_all = select_days(mission)
    gaps = calendar_gaps([g["day"] for g in sel])
    by_version = {v: sum(1 for g in sel if g["version"] == v) for v in PRODUCTS[mission]["versions"]}
    log.info("%s: %d daily files selected, %s to %s, by version %s | %d calendar day(s) have no file at NASA "
             "(longest gap %s)", mission, len(sel), sel[0]["day"], sel[-1]["day"], by_version,
             sum(g[2] for g in gaps), ("%s to %s" % max(gaps, key=lambda g: g[2])[:2]) if gaps else "none")

    name = out_name(mission)
    on_drive = os.path.join(DRIVE_OUT, name)
    local = stage_output(name)
    n = d = q = keep = start = 0
    if os.path.exists(local):
        st = read_state(local)
        if st["mission"] != mission:
            raise PipelineStop(f"{local} was built for {st['mission']}, not {mission}.")
        if np.max(np.abs(np.array(st["bbox"]) - np.array(roi["bbox"]))) > 1e-9:
            raise PipelineStop("The file on Drive was built for a different bounding box. Restore ROI_PAD_DEG "
                               "and the shapefile, or set REBUILD = True.")
        keep, start = resume_point(st, sel)
        d = keep
        n = st["n"] if keep == st["d"] else int(st["first_sounding"][keep])
        q = st["q"] if keep == st["d"] else int(st["first_sequence"][keep])
    todo = sel[start:]
    if not todo:
        log.info("%s: the file already holds every selected day (%d days, %d soundings). Nothing to do.",
                 mission, d, n)
        return dict(path=on_drive, days=d, new_days=0, soundings=n, all_days=n_all)
    log.info("%s: keeping %d day(s) / %d soundings, processing %d day(s) from %s", mission, d, n, len(todo),
             todo[0]["day"])

    ds = create_output(local, mission, roi) if not os.path.exists(local) else raw(nc4.Dataset(local, "a"))
    state = dict(n=n, d=d, q=q, last_time=None)
    tname = nc_name(TIME)
    if n and tname in ds.variables:
        state["last_time"] = float(ds[tname][n - 1])
    commit(ds, n, d, q, False)                          # anything after these markers is being rewritten
    d_start, t_run, last_backup = d, time.time(), time.time()
    trusted = True
    try:
        for b0 in range(0, len(todo), BATCH_DAYS):
            batch = todo[b0:b0 + BATCH_DAYS]
            infos = download_group(batch)
            results = extract_group(client, batch, roi_future, worker_spec(roi))
            n, d, q = write_batch(ds, mission, batch, infos, results, state)
            state.update(n=n, d=d, q=q)
            final = b0 + BATCH_DAYS >= len(todo)
            commit(ds, n, d, q, final)
            remove_raw([g["name"] for g in batch])
            TOTALS[mission + "_files"] = TOTALS.get(mission + "_files", 0) + len(batch)
            log.info("%s %s to %s: %d files -> %d soundings kept (%d inside the polygon, %d file(s) with none) | "
                     "total %d days, %d soundings | %.1f GB free", mission, batch[0]["day"], batch[-1]["day"],
                     len(batch), sum(r["n_kept"] for r in results), sum(r["n_in_polygon"] for r in results),
                     sum(1 for r in results if r["n_kept"] == 0), d, n, free_gb())
            del results
            if not final and time.time() - last_backup >= BACKUP_INTERVAL_S:
                ds.close()
                ds = None
                backup(local, on_drive)
                ds = raw(nc4.Dataset(local, "a"))
                last_backup = time.time()
    except DataIntegrityError:
        trusted = False
        raise
    except (KeyboardInterrupt, PipelineStop) as e:
        log.warning("%s stopped (%s). Saving the %d committed day(s) to Drive.", mission, type(e).__name__, d)
        raise
    finally:
        if ds is not None:
            ds.close()
        if not trusted:
            os.remove(local)                            # the next run starts from the copy on Drive
        elif os.path.exists(local) and d > d_start:
            backup(local, on_drive)
    sizes = read_state(local)["sizes"]
    if sizes["sounding"] > n or sizes["day"] > d:
        log.warning("%s: the file holds %d sounding row(s) after the last committed one (left over from a "
                    "rebuild of changed days). Readers must stop at committed_soundings; set REBUILD = True to "
                    "write a file without them.", mission, sizes["sounding"] - n)
    return dict(path=on_drive, days=d, new_days=d - d_start, soundings=n, all_days=n_all,
                seconds=time.time() - t_run)


def run_all():
    """Run every mission. Stops cleanly on Ctrl-C / interrupt: whatever is committed is copied to Drive."""
    TIMES.clear()
    TOTALS.clear()
    t0 = time.time()
    roi = load_roi()
    ensure_disk_free("start")
    cluster, client = start_cluster()
    out = {}
    try:
        roi_future = client.scatter(roi["wkb"], broadcast=True)      # the polygon travels to the workers once
        for mission in MISSIONS:
            if mission not in PRODUCTS:
                raise PipelineStop(f"MISSIONS names {mission!r}; known missions are {sorted(PRODUCTS)}.")
            out[mission] = run_mission(client, roi, roi_future, mission)
    except KeyboardInterrupt:
        log.warning("Interrupted. Committed data are on Drive; Run all again to continue.")
    except PipelineStop as e:
        log.error("STOPPED: %s", e)
    except DataIntegrityError as e:
        log.error("STOPPED, nothing was copied to Drive from the failed batch: %s", e)
    finally:
        client.close()
        cluster.close()
        shutil.rmtree(DASK_TMP, ignore_errors=True)
    log.info("Run time %.1f min | by stage (s): %s | downloaded %s", (time.time() - t0) / 60,
             {k: round(v) for k, v in TIMES.items()}, human(TOTALS.get("bytes_downloaded", 0)))
    if TEST_MODE:
        for mission, r in out.items():
            if r.get("seconds") and r.get("new_days"):
                size = os.path.getsize(r["path"])
                scale = r["all_days"] / r["new_days"]
                log.info("%s rehearsal: %d days, %d soundings, output %s. Projected whole record (%d days): about "
                         "%.0f min at this speed, output about %s.", mission, r["new_days"], r["soundings"],
                         human(size), r["all_days"], r["seconds"] * scale / 60, human(size * scale))
        log.info("Rehearsal finished. Set TEST_MODE = False in Cell 2 and Run all for the whole record.")
    return out


RESULT = run_all()

In [ ]:
# ═══════════════ CELL 9: audit of the files on Drive + independent re-download check ═══════════════
def audit_output(path):
    """Reopen the copy on Drive and check structure, bookkeeping and order. Returns (info, problems)."""
    problems, info = [], {}
    with nc4.Dataset(path) as ds:
        raw(ds)
        n, d, q = int(ds.committed_soundings), int(ds.committed_days), int(ds.committed_sequences)
        sizes = {k: ds.dimensions[k].size for k in ("sounding", "day", "sequence")}
        if (sizes["sounding"], sizes["day"], sizes["sequence"]) != (n, d, q):
            problems.append(f"rows beyond the commit markers: axes hold {sizes}, committed are "
                            f"{n} soundings / {d} days / {q} sequences")
        cnt = np.asarray(ds["day_n_soundings"][:d], "i8")
        first = np.asarray(ds["day_first_sounding"][:d], "i8")
        if d and (first[0] != 0 or np.any(first[1:] != first[:-1] + cnt[:-1]) or first[-1] + cnt[-1] != n):
            problems.append("the day ledger does not add up to the sounding axis")
        if n:
            di = np.asarray(ds["day_index"][:n], "i8")
            if di.min() < 0 or di.max() >= d or np.any(np.bincount(di, minlength=d) != cnt):
                problems.append("day_index does not agree with the day ledger")
            if np.any(np.diff(di) < 0):
                problems.append("soundings are not grouped by day")
        days = [DAY_EPOCH + timedelta(days=int(x)) for x in ds["day_time"][:d]]
        if any(b <= a for a, b in zip(days, days[1:])):
            problems.append("the day axis is not strictly increasing")
        files = [str(x) for x in ds["day_source_file"][:d]]
        if len(set(files)) != len(files):
            problems.append("a daily file is stored twice")
        svars = [k for k, v in ds.variables.items() if _axis_of(v) == "sounding" and "source_variable" in v.ncattrs()]
        dvars = [k for k, v in ds.variables.items() if _axis_of(v) == "day" and "source_variable" in v.ncattrs()]
        for k, v in ds.variables.items():
            if "long_name" not in v.ncattrs():
                problems.append(f"{k}: no long_name")
            if v.dtype is not str and "_FillValue" not in v.ncattrs() and v.dtype.kind == "f":
                problems.append(f"{k}: no _FillValue")
        order = "n/a"
        if n:
            t = np.asarray(ds[nc_name(TIME)][:n])
            order = "non-decreasing" if np.all(np.diff(t) >= 0) else "NOT in time order"
            if order != "non-decreasing":
                problems.append("soundings are not in time order")
            sid = nc_name("Metadata/SoundingId")
            if sid in ds.variables and np.unique(ds[sid][:n]).size != n:
                problems.append("a sounding identifier occurs more than once")
        empty, second = [], {}
        for k in svars + dvars:
            v = ds[k]
            if v.dtype is str:
                continue
            rows = n if k in svars else d
            if rows and k in svars and np.all(v[:rows] == v.getncattr("_FillValue")):
                empty.append(k)
            if rows and "source_missing_values" in v.ncattrs():
                others = [c for c in np.atleast_1d(v.source_missing_values) if c != v.getncattr("_FillValue")]
                second[k] = int(np.isin(v[:rows], others).sum())

        def hist(name):
            if not n or name not in ds.variables:
                return {}
            u, c = np.unique(ds[name][:n], return_counts=True)
            return {int(a): int(b) for a, b in zip(u, c)}

        versions = PRODUCTS[str(ds.mission)]["versions"]
        gaps = calendar_gaps(days)
        info.update(soundings=n, days=d, sequences=q, first=str(days[0]) if d else None,
                    last=str(days[-1]) if d else None, size=os.path.getsize(path),
                    sounding_variables=len(svars), day_variables=len(dvars),
                    days_without_soundings=int((cnt == 0).sum()), time_order=order,
                    in_polygon=hist("in_polygon").get(1, 0), footprint_in_polygon=hist("footprint_in_polygon").get(1, 0),
                    quality_flag=hist("Quality_Flag"), measurement_mode=hist(nc_name("Metadata/MeasurementMode")),
                    by_version={versions[k]: c for k, c in hist("collection_version").items() if 0 <= k < len(versions)},
                    calendar_days_without_file=sum(g[2] for g in gaps),
                    longest_gaps=[f"{a} to {b} ({c} d)" for a, b, c in sorted(gaps, key=lambda g: -g[2])[:5]],
                    variables_all_missing=empty, values_with_a_second_missing_code=second, processing_status=str(getattr(ds, "processing_status", "?")))
    return info, problems


def _find_granule(short_name, version, name):
    """Look one daily file up again in NASA's catalogue, by name."""
    r = _cmr_get("granules.umm_json", dict(short_name=short_name, version=version, page_size=5,
                                           readable_granule_name=name))
    for it in r.json().get("items", []):
        for u in it["umm"].get("RelatedUrls", []):
            if u.get("Type") == "GET DATA" and u["URL"].startswith("https://") and u["URL"].endswith(name):
                return dict(name=name, url=u["URL"])
    return None


def cross_check(path, k):
    """Download k stored days again, read NASA's variables straight from the fresh files (no worker, no
    batch code) and compare every stored value of those days with them, bit for bit.
    Returns (days checked, values compared, days skipped)."""
    rng = random.Random(20260101)
    tmp = os.path.join(SCRATCH, "recheck")
    os.makedirs(tmp, exist_ok=True)
    checked = values = skipped = 0
    with nc4.Dataset(path) as ds:
        raw(ds)
        d = int(ds.committed_days)
        short = str(ds.product_short_name)
        cnt = np.asarray(ds["day_n_soundings"][:d], "i8")
        first = np.asarray(ds["day_first_sounding"][:d], "i8")
        with_data = np.flatnonzero(cnt > 0).tolist()
        picks = rng.sample(with_data, min(k, len(with_data)))
        for i in sorted(picks):
            name, version = str(ds["day_source_file"][i]), str(ds["day_collection_version"][i])
            g = _find_granule(short, version, name)
            if g is None:
                log.warning("re-check: %s is no longer in NASA's catalogue - skipped", name)
                skipped += 1
                continue
            download_one(g, tmp)
            lo, hi = int(first[i]), int(first[i] + cnt[i])
            src_idx = np.asarray(ds["source_index"][lo:hi], "i8")
            try:
                with nc4.Dataset(os.path.join(tmp, name)) as src:
                    raw(src)
                    for vname, v in ds.variables.items():
                        if "source_variable" not in v.ncattrs() or v.dtype is str:
                            continue
                        p = v.source_variable[1:]
                        axis = _axis_of(v)
                        if axis == "sequence":
                            continue
                        try:
                            sv = src[p]
                        except (IndexError, KeyError):
                            sv = None
                        if axis == "sounding":
                            got = np.asarray(v[lo:hi])
                            want = np.asarray(sv[...])[src_idx] if sv is not None \
                                else _fill_block(v, got.shape)
                        else:
                            got = np.asarray(v[i])
                            want = np.asarray(sv[...]) if sv is not None else _fill_block(v, got.shape)
                        if not _same(got, np.ascontiguousarray(want)):
                            raise DataIntegrityError(f"RE-CHECK FAILED: {vname} of {name} differs from NASA's file.")
                        values += got.size
            finally:
                remove_raw([name], tmp)
            checked += 1
    shutil.rmtree(tmp, ignore_errors=True)
    return checked, values, skipped


def audit_all():
    ok = True
    for mission in MISSIONS:
        path = os.path.join(DRIVE_OUT, out_name(mission))
        if not os.path.exists(path):
            log.warning("AUDIT %s: no file on Drive yet (%s)", mission, path)
            ok = False
            continue
        info, problems = audit_output(path)
        log.info("AUDIT %s: %s", os.path.basename(path), json.dumps(info, default=str))
        try:
            c, v, s = cross_check(path, CROSS_CHECKS)
            log.info("AUDIT %s: %d day(s) downloaded again, %d stored values identical to NASA's files%s", mission,
                     c, v, f" ({s} day(s) skipped)" if s else "")
        except DataIntegrityError as e:
            problems.append(str(e))
        except PipelineStop as e:
            log.warning("AUDIT %s: the re-download check could not run (%s)", mission, e)
        for p in problems:
            log.error("AUDIT %s PROBLEM: %s", mission, p)
        ok = ok and not problems
        if info.get("processing_status") != "complete":
            log.warning("AUDIT %s: the file is not complete yet - Run all again to continue.", mission)
    log.info("AUDIT RESULT: %s", "no problem found" if ok else "PROBLEMS FOUND - see the lines above")
    return ok


AUDIT_OK = audit_all()